In [2]:
import cv2
import mediapipe as mp
import numpy as np
import os


In [8]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options,
                                       num_hands=1,
                                       min_hand_detection_confidence=0.5)
detector = vision.HandLandmarker.create_from_options(options)


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1
)

with HandLandmarker.create_from_options(options) as landmarker:

    image = cv2.imread("C:\Projects\signia-fsl-recognition\static_raw\B_85.jpg.")
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    result = landmarker.detect(mp_image)

    if result.hand_landmarks:
        landmarks = []
        for lm in result.hand_landmarks[0]:
            landmarks.extend([lm.x, lm.y, lm.z])

        landmarks = np.array(landmarks)
        print(landmarks.shape)  # Should print (63,)


(63,)


Step 2.2 — Extract Landmarks from ONE Image

In [12]:
def normalize_landmarks(landmarks):
    landmarks = landmarks.reshape(21, 3)
    
    wrist = landmarks[0]
    landmarks = landmarks - wrist
    
    max_value = np.max(np.abs(landmarks))
    if max_value != 0:
        landmarks = landmarks / max_value
    
    return landmarks.flatten()


Loop all alphabets


In [14]:
import os

input_dir = "static_raw"
output_dir = "static_landmarks"

os.makedirs(output_dir, exist_ok=True)

with HandLandmarker.create_from_options(options) as landmarker:
    
    for label in os.listdir(input_dir):
        label_path = os.path.join(input_dir, label)
        
        if not os.path.isdir(label_path):
            continue
        
        print(f"Processing {label}...")
        
        save_label_path = os.path.join(output_dir, label)
        os.makedirs(save_label_path, exist_ok=True)
        
        for img_name in os.listdir(label_path):
            
            img_path = os.path.join(label_path, img_name)
            image = cv2.imread(img_path)
            
            if image is None:
                continue
            
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=image_rgb
            )
            
            result = landmarker.detect(mp_image)
            
            if result.hand_landmarks:
                landmarks = []
                
                for lm in result.hand_landmarks[0]:
                    landmarks.extend([lm.x, lm.y, lm.z])
                
                landmarks = np.array(landmarks)
                landmarks = normalize_landmarks(landmarks)
                
                save_path = os.path.join(
                    save_label_path,
                    img_name.replace(".jpg", ".npy")
                )
                
                np.save(save_path, landmarks)
        
        print(f"{label} done.")


Processing A...
A done.
Processing B...
B done.
Processing C...
C done.
Processing D...
D done.
Processing E...
E done.
Processing F...
F done.
Processing G...
G done.
Processing H...
H done.
Processing I...
I done.
Processing J...
J done.
Processing K...
K done.
Processing L...
L done.
Processing M...
M done.
Processing N...
N done.
Processing O...
O done.
Processing P...
P done.
Processing Q...
Q done.
Processing R...
R done.
Processing S...
S done.
Processing T...
T done.
Processing U...
U done.
Processing V...
V done.
Processing W...
W done.
Processing X...
X done.
Processing Y...
Y done.
Processing Z...
Z done.


In [15]:
sample = np.load("static_landmarks/A/1_jpg.rf.3443d1521395d39ab2f4dff1517cae69.npy")
print(sample.shape)
    

(63,)
